# Rembrandt Authentication Prototype

**Question:** Does DINOv2 (pre-trained, no fine-tuning) capture style differences between confirmed Rembrandt autographs and "circle of" / "school of" works?

**Data:** Rijksmuseum APIs (no auth needed) — Search API + IIIF Image API + Linked Data

**Compute:** Google Colab Pro (T4 GPU)

In [ ]:
# Cell 0: Setup & Config
# Install deps not pre-installed on Colab
!pip install -q umap-learn faiss-cpu

import json
import os
import re
import time
import warnings
from pathlib import Path
from urllib.parse import urlencode, quote_plus

import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image
from io import BytesIO

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Constants ---
SEARCH_URL = "https://data.rijksmuseum.nl/search/collection"
DATA_URL = "https://data.rijksmuseum.nl"  # /{id}?_profile=la-framed or edm-framed
IIIF_BASE = "https://iiif.micr.io"  # /{identifier}/full/!2000,2000/0/default.jpg

CACHE_DIR = Path("/content/cache")
CACHE_META = CACHE_DIR / "metadata"
CACHE_IMG = CACHE_DIR / "images"
CACHE_EMB = CACHE_DIR / "embeddings"
for d in [CACHE_META, CACHE_IMG, CACHE_EMB]:
    d.mkdir(parents=True, exist_ok=True)

RATE_LIMIT = 0.3  # seconds between API calls
TILE_SIZE = 224
IMG_MAX_PX = 2000
CHUNK_SIZE = 20  # paintings per processing chunk
BATCH_SIZE = 32  # tiles per GPU batch
EMBED_DIM = 768  # DINOv2 ViT-B/14

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


# --- Helpers ---
def fetch_with_retry(url, params=None, headers=None, max_retries=3, backoff=0.3):
    """GET with exponential backoff. Returns response or None."""
    for attempt in range(max_retries):
        try:
            time.sleep(backoff * (2 ** attempt) if attempt > 0 else RATE_LIMIT)
            resp = requests.get(url, params=params, headers=headers, timeout=30)
            if resp.status_code == 200:
                return resp
            if resp.status_code == 429:  # rate limited
                print(f"  Rate limited, backing off...")
                continue
            print(f"  HTTP {resp.status_code} for {url}")
            return None
        except requests.RequestException as e:
            print(f"  Request failed (attempt {attempt+1}): {e}")
    print(f"  All retries exhausted for {url}")
    return None


def fetch_json(url, params=None, headers=None):
    """Fetch and parse JSON. Returns dict or None."""
    resp = fetch_with_retry(url, params=params, headers=headers)
    if resp is None:
        return None
    try:
        return resp.json()
    except json.JSONDecodeError:
        print(f"  Invalid JSON from {url}")
        return None


print("Setup complete.")

In [ ]:
# Cell 1: API Exploration — Discover JSON structure, build parsers
# Run this cell once to understand the API response format.

# --- Resolve a few known Rembrandt paintings ---
SAMPLE_IDS = [
    "200106038",  # Landscape with a Stone Bridge
    "200107928",  # The Night Watch
]

print("=" * 60)
print("LINKED ART PROFILE (la-framed) — metadata structure")
print("=" * 60)
sample_la = fetch_json(f"{DATA_URL}/{SAMPLE_IDS[0]}", params={"_profile": "la-framed"})
if sample_la:
    print(json.dumps(sample_la, indent=2, ensure_ascii=False)[:3000])

print("\n" + "=" * 60)
print("EDM PROFILE (edm-framed) — for IIIF image URL")
print("=" * 60)
sample_edm = fetch_json(f"{DATA_URL}/{SAMPLE_IDS[0]}", params={"_profile": "edm-framed"})
if sample_edm:
    print(json.dumps(sample_edm, indent=2, ensure_ascii=False)[:3000])


# --- Attribution classification ---
ATTRIBUTION_PATTERNS = {
    "autograph": [],
    "workshop": [r"\bworkshop\s+of\b", r"\batelier\s+van\b"],
    "circle": [r"\bcircle\s+of\b", r"\bomgeving\s+van\b", r"\bkring\s+van\b"],
    "school": [r"\bschool\s+of\b", r"\bschool\s+van\b"],
    "style": [r"\bstyle\s+of\b", r"\bstijl\s+van\b", r"\bmanner\s+of\b",
              r"\bmanier\s+van\b", r"\bnavolger\b", r"\bfollower\b"],
    "after": [r"\bafter\b", r"\bnaar\b", r"\bcopy\s+after\b", r"\bkopie\s+naar\b"],
    "attributed": [r"\battributed\s+to\b", r"\btoegeschreven\s+aan\b"],
}


def classify_attribution(creator_str, title_str=""):
    """Classify attribution level from creator/title strings."""
    text = f"{creator_str} {title_str}".lower()
    for level, patterns in ATTRIBUTION_PATTERNS.items():
        if level == "autograph":
            continue
        for pat in patterns:
            if re.search(pat, text):
                return level
    return "autograph"


def extract_iiif_id_from_edm(edm_json):
    """Extract the Micrio IIIF identifier from EDM JSON-LD."""
    if not edm_json:
        return None
    text = json.dumps(edm_json)
    match = re.search(r'iiif\.micr\.io/([a-zA-Z0-9]+)', text)
    return match.group(1) if match else None


def parse_la_metadata(la_json):
    """Extract title, creator, date from Linked Art JSON-LD.

    Actual structure (discovered from API):
    - Title: identified_by[].content where type=Name (or top-level _label)
    - Creator: produced_by.referred_to_by[].content (AAT 300435416)
    - Date: produced_by.timespan.identified_by[].content (type Name)
    """
    if not la_json:
        return {}

    result = {}

    # Title — try _label first, then identified_by with type Name
    if "_label" in la_json:
        result["title"] = la_json["_label"]
    if "title" not in result:
        for ident in la_json.get("identified_by", []):
            if isinstance(ident, dict) and ident.get("type") == "Name":
                content = ident.get("content", "")
                if content:
                    result["title"] = content
                    break

    produced = la_json.get("produced_by", {})
    if not isinstance(produced, dict):
        return result

    # Creator — from referred_to_by with AAT 300435416
    for ref in produced.get("referred_to_by", []):
        if not isinstance(ref, dict):
            continue
        for cls in ref.get("classified_as", []):
            if isinstance(cls, dict) and "300435416" in cls.get("id", ""):
                result["creator"] = ref.get("content", "")
                break
        if "creator" in result:
            break

    # Creator fallback — produced_by.part[].carried_out_by[]._label
    if "creator" not in result:
        for part in produced.get("part", []):
            if not isinstance(part, dict):
                continue
            for person in part.get("carried_out_by", []):
                if isinstance(person, dict) and "_label" in person:
                    result["creator"] = person["_label"]
                    break
            if "creator" in result:
                break

    # Date — timespan.identified_by[].content (prefer English aat/300388277)
    ts = produced.get("timespan", {})
    if isinstance(ts, dict):
        for ident in ts.get("identified_by", []):
            if isinstance(ident, dict) and ident.get("type") == "Name":
                content = ident.get("content", "")
                if content:
                    result["date"] = content
                    for lang in ident.get("language", []):
                        if isinstance(lang, dict) and "300388277" in lang.get("id", ""):
                            result["date"] = content
                            break

    return result


def extract_title_from_edm(edm_json):
    """Fallback: extract title from EDM JSON-LD via dc:title or dcterms:title."""
    if not edm_json:
        return None
    text = json.dumps(edm_json, ensure_ascii=False)
    # Look for dc:title or title fields with @value
    for key in ["dc:title", "dcterms:title", "title"]:
        match = re.search(rf'"{key}"[^"]*"@value"\s*:\s*"([^"]+)"', text)
        if match:
            return match.group(1)
    return None


# --- Test parsers on samples ---
print("\n" + "=" * 60)
print("PARSER TEST")
print("=" * 60)
for obj_id in SAMPLE_IDS:
    la = fetch_json(f"{DATA_URL}/{obj_id}", params={"_profile": "la-framed"})
    edm = fetch_json(f"{DATA_URL}/{obj_id}", params={"_profile": "edm-framed"})
    meta = parse_la_metadata(la)
    # Fallback title from EDM
    if "title" not in meta:
        edm_title = extract_title_from_edm(edm)
        if edm_title:
            meta["title"] = edm_title
    iiif_id = extract_iiif_id_from_edm(edm)
    attrib = classify_attribution(meta.get("creator", ""), meta.get("title", ""))
    print(f"\n  ID: {obj_id}")
    print(f"  Title: {meta.get('title', '?')}")
    print(f"  Creator: {meta.get('creator', '?')}")
    print(f"  Date: {meta.get('date', '?')}")
    print(f"  IIIF ID: {iiif_id}")
    print(f"  Attribution: {attrib}")
    if iiif_id:
        print(f"  Image URL: {IIIF_BASE}/{iiif_id}/full/!{IMG_MAX_PX},{IMG_MAX_PX}/0/default.jpg")

print("\nAPI exploration complete. Check output above and adjust parsers if needed.")

In [ ]:
# Cell 2: Build Painting Inventory
# Search API for Rembrandt + circle + control group, resolve metadata, output CSV

CSV_PATH = CACHE_META / "painting_inventory.csv"

# Check cache first
if CSV_PATH.exists():
    inventory = pd.read_csv(CSV_PATH)
    print(f"Loaded cached inventory: {len(inventory)} paintings")
    print(inventory["artist_group"].value_counts())
else:
    QUERIES = [
        # Rembrandt autographs
        ({"creator": "Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_autograph", "autograph"),
        # Workshop / Circle / School of Rembrandt
        ({"creator": "Workshop of Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_circle", "workshop"),
        ({"creator": "Circle of Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_circle", "circle"),
        ({"creator": "School of Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_circle", "school"),
        ({"creator": "Style of Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_circle", "style"),
        ({"creator": "Follower of Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_circle", "style"),
        ({"creator": "After Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_circle", "after"),
        ({"creator": "Attributed to Rembrandt van Rijn", "type": "painting", "imageAvailable": "true"},
         "rembrandt_circle", "attributed"),
        # Rembrandt pupils (control)
        ({"creator": "Ferdinand Bol", "type": "painting", "imageAvailable": "true"},
         "rembrandt_pupil", "autograph"),
        ({"creator": "Govert Flinck", "type": "painting", "imageAvailable": "true"},
         "rembrandt_pupil", "autograph"),
        ({"creator": "Jan Lievens", "type": "painting", "imageAvailable": "true"},
         "rembrandt_pupil", "autograph"),
        # Non-circle Dutch masters (control)
        ({"creator": "Frans Hals", "type": "painting", "imageAvailable": "true"},
         "dutch_other", "autograph"),
        ({"creator": "Johannes Vermeer", "type": "painting", "imageAvailable": "true"},
         "dutch_other", "autograph"),
    ]

    def search_all_pages(params):
        """Fetch all pages from search API. Returns list of object IDs."""
        ids = []
        page_params = dict(params)
        while True:
            data = fetch_json(SEARCH_URL, params=page_params)
            if not data:
                break
            for item in data.get("orderedItems", []):
                match = re.search(r'/(\d+)$', item.get("id", ""))
                if match:
                    ids.append(match.group(1))
            next_page = data.get("next", {})
            next_url = next_page.get("id", "") if isinstance(next_page, dict) else ""
            if not next_url:
                break
            token_match = re.search(r'pageToken=([^&]+)', next_url)
            if token_match:
                page_params["pageToken"] = token_match.group(1)
            else:
                break
        return ids

    # --- Collect all painting IDs ---
    print("Searching for paintings...")
    all_records = []
    seen_ids = set()

    for params, artist_group, force_attrib in QUERIES:
        creator = params["creator"]
        ids = search_all_pages(params)
        new_ids = [i for i in ids if i not in seen_ids]
        seen_ids.update(new_ids)
        print(f"  {creator}: {len(ids)} found, {len(new_ids)} new")
        for obj_id in new_ids:
            all_records.append({
                "obj_id": obj_id,
                "search_creator": creator,
                "artist_group": artist_group,
                "force_attribution": force_attrib,
            })

    print(f"\nTotal unique paintings: {len(all_records)}")

    # --- Resolve metadata for each painting ---
    print("\nResolving metadata (this takes a few minutes)...")
    rows = []
    for i, rec in enumerate(all_records):
        obj_id = rec["obj_id"]

        # Check individual cache
        cache_file = CACHE_META / f"{obj_id}.json"
        if cache_file.exists():
            cached = json.loads(cache_file.read_text())
            rows.append(cached)
            if (i + 1) % 50 == 0:
                print(f"  {i+1}/{len(all_records)} (cached)")
            continue

        # Fetch LA metadata
        la = fetch_json(f"{DATA_URL}/{obj_id}", params={"_profile": "la-framed"})
        meta = parse_la_metadata(la)

        # Fetch EDM for IIIF ID (and fallback title)
        edm = fetch_json(f"{DATA_URL}/{obj_id}", params={"_profile": "edm-framed"})
        iiif_id = extract_iiif_id_from_edm(edm)
        if "title" not in meta:
            edm_title = extract_title_from_edm(edm)
            if edm_title:
                meta["title"] = edm_title

        creator = meta.get("creator", "") or rec["search_creator"]
        title = meta.get("title", "")
        attribution = rec["force_attribution"] or classify_attribution(creator, title)

        row = {
            "obj_id": obj_id,
            "title": title,
            "creator": creator,
            "date": meta.get("date", ""),
            "iiif_id": iiif_id,
            "artist_group": rec["artist_group"],
            "attribution": attribution,
        }
        rows.append(row)
        cache_file.write_text(json.dumps(row, ensure_ascii=False))

        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(all_records)} resolved")

    # --- Build DataFrame, filter, save ---
    inventory = pd.DataFrame(rows)
    before = len(inventory)
    inventory = inventory[inventory["iiif_id"].notna() & (inventory["iiif_id"] != "")].copy()
    inventory = inventory.drop_duplicates(subset="obj_id").reset_index(drop=True)
    print(f"\nFiltered: {before} → {len(inventory)} paintings with IIIF images")
    inventory.to_csv(CSV_PATH, index=False)
    print(f"Saved to {CSV_PATH}")

# --- Summary ---
print(f"\n{'='*60}")
print("INVENTORY SUMMARY")
print(f"{'='*60}")
print(f"Total paintings: {len(inventory)}")
print(f"\nBy artist group:")
print(inventory["artist_group"].value_counts().to_string())
print(f"\nBy attribution:")
print(inventory["attribution"].value_counts().to_string())
print(f"\nSample rows:")
inventory.head(10)

In [ ]:
# Cell 3: Download Images & Tile
# Download via IIIF at 2000px, tile into 224×224 patches
# Process in chunks of 20 paintings to stay within Colab RAM

from torchvision import transforms

# DINOv2 preprocessing (ImageNet normalization)
dino_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def download_image(iiif_id):
    """Download image via IIIF. Returns PIL Image or None. Caches to disk."""
    cache_path = CACHE_IMG / f"{iiif_id}.jpg"
    if cache_path.exists():
        try:
            return Image.open(cache_path).convert("RGB")
        except Exception:
            cache_path.unlink(missing_ok=True)

    url = f"{IIIF_BASE}/{iiif_id}/full/!{IMG_MAX_PX},{IMG_MAX_PX}/0/default.jpg"
    resp = fetch_with_retry(url)
    if resp is None:
        return None
    try:
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img.save(cache_path, "JPEG", quality=95)
        return img
    except Exception as e:
        print(f"  Failed to decode image {iiif_id}: {e}")
        return None


def tile_image(img, tile_size=TILE_SIZE):
    """Tile PIL image into non-overlapping patches. Returns list of PIL images."""
    w, h = img.size
    tiles = []
    for y in range(0, h - tile_size + 1, tile_size):
        for x in range(0, w - tile_size + 1, tile_size):
            tile = img.crop((x, y, x + tile_size, y + tile_size))
            tiles.append(tile)
    return tiles


def prepare_tile_batch(tiles):
    """Convert list of PIL tiles to a batched tensor."""
    tensors = [dino_transform(t) for t in tiles]
    return torch.stack(tensors)


# --- Download and report ---
print("Downloading images...")
download_stats = {"success": 0, "fail": 0, "cached": 0}

for i, row in inventory.iterrows():
    iiif_id = row["iiif_id"]
    cache_path = CACHE_IMG / f"{iiif_id}.jpg"
    if cache_path.exists():
        download_stats["cached"] += 1
        continue
    img = download_image(iiif_id)
    if img:
        download_stats["success"] += 1
    else:
        download_stats["fail"] += 1
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(inventory)} downloaded")

total_ok = download_stats["success"] + download_stats["cached"]
print(f"\nDownload complete: {total_ok} available "
      f"({download_stats['cached']} cached, {download_stats['success']} new), "
      f"{download_stats['fail']} failed")

# --- Show tile counts for a sample ---
sample_iiif = inventory.iloc[0]["iiif_id"]
sample_img = download_image(sample_iiif)
if sample_img:
    tiles = tile_image(sample_img)
    print(f"\nSample: {inventory.iloc[0]['title']}")
    print(f"  Image size: {sample_img.size}")
    print(f"  Tiles: {len(tiles)} ({TILE_SIZE}×{TILE_SIZE})")
    del tiles  # free memory

print("\nImages ready for embedding.")

In [ ]:
# Cell 4: DINOv2 Embedding
# Extract CLS + mean patch tokens per tile, aggregate per painting
# Chunked processing: 20 paintings at a time, batch size 32 on T4

EMB_PATH = CACHE_EMB / "embeddings.npz"

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    painting_ids = data["painting_ids"]
    cls_embeddings = data["cls_embeddings"]
    patch_embeddings = data["patch_embeddings"]
    artist_groups = data["artist_groups"]
    attributions = data["attributions"]
    print(f"Loaded cached embeddings: {len(painting_ids)} paintings, "
          f"{cls_embeddings.shape[1]}d CLS + {patch_embeddings.shape[1]}d patch")
else:
    # Load DINOv2
    print("Loading DINOv2 ViT-B/14...")
    model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
    model = model.to(DEVICE)
    model.eval()
    print(f"Model loaded on {DEVICE}")

    @torch.no_grad()
    def embed_tiles(tile_batch_tensor):
        """Embed a batch of tiles. Returns (cls_tokens, patch_means) as numpy."""
        x = tile_batch_tensor.to(DEVICE)
        # DINOv2 forward_features returns dict with keys:
        #   x_norm_clstoken (B, D), x_norm_patchtokens (B, N, D)
        out = model.forward_features(x)
        cls_tok = out["x_norm_clstoken"].cpu().numpy()  # (B, 768)
        patch_tok = out["x_norm_patchtokens"].mean(dim=1).cpu().numpy()  # (B, 768)
        return cls_tok, patch_tok

    # --- Process in chunks ---
    painting_ids = []
    cls_embeddings = []
    patch_embeddings = []
    artist_groups_list = []
    attributions_list = []
    skipped = 0

    total = len(inventory)
    chunk_starts = list(range(0, total, CHUNK_SIZE))
    print(f"\nEmbedding {total} paintings in {len(chunk_starts)} chunks of {CHUNK_SIZE}...")

    for ci, start in enumerate(chunk_starts):
        chunk = inventory.iloc[start:start + CHUNK_SIZE]

        for _, row in chunk.iterrows():
            iiif_id = row["iiif_id"]
            img = download_image(iiif_id)
            if img is None:
                skipped += 1
                continue

            tiles = tile_image(img)
            if not tiles:
                skipped += 1
                continue

            # Embed tiles in batches
            all_cls = []
            all_patch = []
            for b in range(0, len(tiles), BATCH_SIZE):
                batch_tiles = tiles[b:b + BATCH_SIZE]
                batch_tensor = prepare_tile_batch(batch_tiles)
                cls_tok, patch_tok = embed_tiles(batch_tensor)
                all_cls.append(cls_tok)
                all_patch.append(patch_tok)

            # Aggregate: mean over all tiles → one embedding per painting
            all_cls = np.concatenate(all_cls, axis=0)
            all_patch = np.concatenate(all_patch, axis=0)
            painting_cls = all_cls.mean(axis=0)  # (768,)
            painting_patch = all_patch.mean(axis=0)  # (768,)

            painting_ids.append(row["obj_id"])
            cls_embeddings.append(painting_cls)
            patch_embeddings.append(painting_patch)
            artist_groups_list.append(row["artist_group"])
            attributions_list.append(row["attribution"])

            # Free tile memory
            del tiles, all_cls, all_patch

        print(f"  Chunk {ci+1}/{len(chunk_starts)} done "
              f"({len(painting_ids)} embedded, {skipped} skipped)")

    # Convert to arrays
    painting_ids = np.array(painting_ids)
    cls_embeddings = np.array(cls_embeddings)
    patch_embeddings = np.array(patch_embeddings)
    artist_groups = np.array(artist_groups_list)
    attributions = np.array(attributions_list)

    # Save
    np.savez(EMB_PATH,
             painting_ids=painting_ids,
             cls_embeddings=cls_embeddings,
             patch_embeddings=patch_embeddings,
             artist_groups=artist_groups,
             attributions=attributions)
    print(f"\nSaved embeddings to {EMB_PATH}")

    # Cleanup
    del model
    torch.cuda.empty_cache()

# Full embedding = concat CLS + patch (1536d)
full_embeddings = np.concatenate([cls_embeddings, patch_embeddings], axis=1)

print(f"\nEmbedding summary:")
print(f"  Paintings: {len(painting_ids)}")
print(f"  CLS dim: {cls_embeddings.shape[1]}")
print(f"  Patch dim: {patch_embeddings.shape[1]}")
print(f"  Full dim: {full_embeddings.shape[1]}")
print(f"\n  Groups: {dict(zip(*np.unique(artist_groups, return_counts=True)))}")

In [ ]:
# Cell 5: Analysis — "Does the Signal Exist?"
# Five visualizations, three quantitative tests

import matplotlib.pyplot as plt
import umap
from scipy.stats import mannwhitneyu
from sklearn.metrics.pairwise import cosine_similarity

# Color map for artist groups
GROUP_COLORS = {
    "rembrandt_autograph": "#e41a1c",  # red
    "rembrandt_circle": "#ff7f00",     # orange
    "rembrandt_pupil": "#984ea3",      # purple
    "dutch_other": "#4daf4a",          # green
}
GROUP_LABELS = {
    "rembrandt_autograph": "Rembrandt (autograph)",
    "rembrandt_circle": "Circle/Workshop/School",
    "rembrandt_pupil": "Pupils (Bol, Flinck, Lievens)",
    "dutch_other": "Other Dutch (Hals, Vermeer)",
}

# Build lookup from painting_id to inventory row
id_to_row = {}
for _, row in inventory.iterrows():
    id_to_row[str(row["obj_id"])] = row

# ============================================================
# 5a: UMAP scatter (color = artist_group)
# ============================================================
print("Computing UMAP projection...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
umap_coords = reducer.fit_transform(full_embeddings)

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
for group, color in GROUP_COLORS.items():
    mask = artist_groups == group
    if mask.sum() == 0:
        continue
    ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
               c=color, label=GROUP_LABELS[group], s=60, alpha=0.7, edgecolors="white", linewidths=0.5)
ax.set_title("DINOv2 Embedding Space — UMAP Projection", fontsize=14)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.legend(fontsize=10, loc="best")
plt.tight_layout()
plt.savefig(CACHE_EMB / "umap_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("5a: UMAP scatter complete.\n")

# ============================================================
# 5b: Cosine similarity distributions + Mann-Whitney U
# ============================================================
print("Computing cosine similarities...")
sim_matrix = cosine_similarity(full_embeddings)

# Masks
auto_mask = artist_groups == "rembrandt_autograph"
circle_mask = artist_groups == "rembrandt_circle"
pupil_mask = artist_groups == "rembrandt_pupil"
other_mask = artist_groups == "dutch_other"

def pairwise_sims(mask_a, mask_b, sim_mat):
    """Extract upper-triangle pairwise similarities between two groups."""
    idx_a = np.where(mask_a)[0]
    idx_b = np.where(mask_b)[0]
    sims = []
    for i in idx_a:
        for j in idx_b:
            if i < j or not np.array_equal(mask_a, mask_b):
                sims.append(sim_mat[i, j])
    return np.array(sims)

# Intra-autograph similarities
auto_auto = pairwise_sims(auto_mask, auto_mask, sim_matrix)
# Autograph vs circle
auto_circle = pairwise_sims(auto_mask, circle_mask, sim_matrix)
# Autograph vs pupils
auto_pupil = pairwise_sims(auto_mask, pupil_mask, sim_matrix)
# Autograph vs other Dutch
auto_other = pairwise_sims(auto_mask, other_mask, sim_matrix)

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
bins = np.linspace(0, 1, 50)
if len(auto_auto) > 0:
    ax.hist(auto_auto, bins=bins, alpha=0.6, label=f"Autograph↔Autograph (n={len(auto_auto)})", color="#e41a1c")
if len(auto_circle) > 0:
    ax.hist(auto_circle, bins=bins, alpha=0.6, label=f"Autograph↔Circle (n={len(auto_circle)})", color="#ff7f00")
if len(auto_pupil) > 0:
    ax.hist(auto_pupil, bins=bins, alpha=0.6, label=f"Autograph↔Pupils (n={len(auto_pupil)})", color="#984ea3")
if len(auto_other) > 0:
    ax.hist(auto_other, bins=bins, alpha=0.6, label=f"Autograph↔Other Dutch (n={len(auto_other)})", color="#4daf4a")
ax.set_title("Cosine Similarity Distributions — THE MONEY PLOT", fontsize=14)
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Count")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(CACHE_EMB / "cosine_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# Mann-Whitney U tests
print("Mann-Whitney U tests (one-tailed: intra > inter):")
for name, inter_sims in [("circle", auto_circle), ("pupils", auto_pupil), ("other", auto_other)]:
    if len(auto_auto) > 0 and len(inter_sims) > 0:
        stat, p = mannwhitneyu(auto_auto, inter_sims, alternative="greater")
        effect = auto_auto.mean() - inter_sims.mean()
        print(f"  Autograph↔Autograph vs Autograph↔{name}: "
              f"U={stat:.0f}, p={p:.2e}, Δmean={effect:+.4f}")
print("5b: Cosine distributions complete.\n")

# ============================================================
# 5c: Cosine similarity heatmap (ordered by group)
# ============================================================
group_order = ["rembrandt_autograph", "rembrandt_circle", "rembrandt_pupil", "dutch_other"]
sort_idx = np.argsort([group_order.index(g) for g in artist_groups])
sorted_sim = sim_matrix[sort_idx][:, sort_idx]
sorted_groups = artist_groups[sort_idx]

# Group boundaries for lines
boundaries = []
for g in group_order:
    count = (sorted_groups == g).sum()
    boundaries.append(count)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(sorted_sim, cmap="RdYlBu_r", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Cosine Similarity")

# Draw group boundaries
cumsum = np.cumsum(boundaries[:-1])
for b in cumsum:
    ax.axhline(y=b - 0.5, color="black", linewidth=1)
    ax.axvline(x=b - 0.5, color="black", linewidth=1)

# Label groups
positions = []
pos = 0
for g, count in zip(group_order, boundaries):
    if count > 0:
        positions.append((pos + count / 2, GROUP_LABELS.get(g, g)))
    pos += count
ax.set_xticks([p for p, _ in positions])
ax.set_xticklabels([l for _, l in positions], rotation=45, ha="right", fontsize=9)
ax.set_yticks([p for p, _ in positions])
ax.set_yticklabels([l for _, l in positions], fontsize=9)
ax.set_title("Cosine Similarity Heatmap (Block-Diagonal Structure?)", fontsize=14)
plt.tight_layout()
plt.savefig(CACHE_EMB / "cosine_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("5c: Heatmap complete.\n")

# ============================================================
# 5d: KNN leave-one-out classification
# ============================================================
print("KNN leave-one-out classification...")

# Binary task: rembrandt_autograph vs everything else
labels = (artist_groups == "rembrandt_autograph").astype(int)
baseline_acc = max(labels.mean(), 1 - labels.mean())
print(f"  Baseline (majority class): {baseline_acc:.1%}")

for K in [3, 5, 7]:
    correct = 0
    for i in range(len(full_embeddings)):
        # Compute distances to all others
        sims = sim_matrix[i].copy()
        sims[i] = -1  # exclude self
        neighbors = np.argsort(sims)[-K:]  # top K most similar
        vote = labels[neighbors].mean() > 0.5
        if vote == labels[i]:
            correct += 1
    acc = correct / len(labels)
    print(f"  K={K}: {acc:.1%} accuracy ({correct}/{len(labels)})")
print("5d: KNN classification complete.\n")

# ============================================================
# 5e: Top 10 "circle of" paintings closest to autograph centroid
# ============================================================
print("Top 10 circle/workshop paintings closest to autograph centroid:")

auto_centroid = full_embeddings[auto_mask].mean(axis=0, keepdims=True)
circle_idx = np.where(circle_mask)[0]

if len(circle_idx) > 0:
    circle_sims = cosine_similarity(full_embeddings[circle_idx], auto_centroid).flatten()
    top_10_idx = np.argsort(circle_sims)[-10:][::-1]

    print(f"\n{'Rank':<5} {'Sim':>6} {'Title':<50} {'Creator'}")
    print("-" * 90)
    for rank, idx in enumerate(top_10_idx, 1):
        global_idx = circle_idx[idx]
        pid = str(painting_ids[global_idx])
        sim = circle_sims[idx]
        row_data = id_to_row.get(pid, {})
        title = row_data.get("title", "?")[:48] if hasattr(row_data, "get") else "?"
        creator = row_data.get("creator", "?") if hasattr(row_data, "get") else "?"
        print(f"  {rank:<3} {sim:>6.4f} {title:<50} {creator}")
else:
    print("  No circle/workshop paintings found.")

print("\n5e: Candidate list complete.")

In [ ]:
# Cell 6: Results Summary — Decision Framework

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

# Gather metrics
auto_auto_mean = auto_auto.mean() if len(auto_auto) > 0 else float("nan")
auto_circle_mean = auto_circle.mean() if len(auto_circle) > 0 else float("nan")
auto_other_mean = auto_other.mean() if len(auto_other) > 0 else float("nan")

# Best p-value from Mann-Whitney tests
p_values = {}
for name, inter_sims in [("circle", auto_circle), ("pupils", auto_pupil), ("other", auto_other)]:
    if len(auto_auto) > 0 and len(inter_sims) > 0:
        _, p = mannwhitneyu(auto_auto, inter_sims, alternative="greater")
        p_values[name] = p

# KNN accuracy at K=5
knn_correct = 0
for i in range(len(full_embeddings)):
    sims = sim_matrix[i].copy()
    sims[i] = -1
    neighbors = np.argsort(sims)[-5:]
    vote = labels[neighbors].mean() > 0.5
    if vote == labels[i]:
        knn_correct += 1
knn_acc = knn_correct / len(labels)

print(f"\nKey Metrics:")
print(f"  Paintings embedded: {len(painting_ids)}")
print(f"  Autograph↔Autograph mean sim: {auto_auto_mean:.4f}")
print(f"  Autograph↔Circle mean sim:    {auto_circle_mean:.4f}")
print(f"  Autograph↔Other mean sim:     {auto_other_mean:.4f}")
print(f"  Separation (auto-auto minus auto-circle): {auto_auto_mean - auto_circle_mean:+.4f}")
for name, p in p_values.items():
    print(f"  Mann-Whitney p ({name}): {p:.2e}")
print(f"  KNN K=5 accuracy: {knn_acc:.1%}")
print(f"  Baseline (majority): {baseline_acc:.1%}")

# Decision
print(f"\n{'='*60}")
print("DECISION FRAMEWORK")
print(f"{'='*60}")
print("""
| Signal   | Criteria                                        | Next Step                              |
|----------|-------------------------------------------------|----------------------------------------|
| STRONG   | UMAP separates, p < 0.01, KNN > 85%            | Scale to multi-artist, multi-collection |
| WEAK     | Partial separation, p < 0.05, KNN 75-85%       | Try full-res IIIF tiles or fine-tune   |
| NONE     | No separation, p > 0.05, KNN ~75%              | Try custom CNN or CLIP with prompts    |
""")

# Auto-assess
min_p = min(p_values.values()) if p_values else 1.0
if min_p < 0.01 and knn_acc > 0.85:
    signal = "STRONG"
elif min_p < 0.05 and knn_acc > baseline_acc:
    signal = "WEAK"
else:
    signal = "NONE"

print(f"Assessment: **{signal}** signal detected")
print(f"  min p-value: {min_p:.2e} ({'< 0.01' if min_p < 0.01 else '< 0.05' if min_p < 0.05 else '>= 0.05'})")
print(f"  KNN accuracy: {knn_acc:.1%} vs {baseline_acc:.1%} baseline")

if signal == "STRONG":
    print("\n→ Proceed to Phase 2: Multi-artist style embedding space")
    print("  - Add contrastive/metric learning on top of DINOv2")
    print("  - Build reference signatures for 200-500 artists")
    print("  - Stream 500K+ paintings via IIIF")
elif signal == "WEAK":
    print("\n→ Try improvements before scaling:")
    print("  - Use full-resolution IIIF tiles (not downscaled)")
    print("  - Fine-tune linear probe on top of DINOv2 features")
    print("  - Weight tiles by entropy/detail (skip blank backgrounds)")
elif signal == "NONE":
    print("\n→ Pivot approach:")
    print("  - Try CLIP with style-descriptive text prompts")
    print("  - Train custom CNN on authentication task")
    print("  - Consider that generic vision embeddings may not capture 'style'")

print(f"\n{'='*60}")
print("Prototype complete. Review plots above for qualitative assessment.")
print(f"{'='*60}")